In [ ]:
exit()

In [ ]:
def make_intervention_hook(direction, alpha):
    """
    direction: [d_model] tensor (on same device)
    alpha: float
    """
    def hook_fn(activations, hook):
        # activations: [batch, seq, d_model]
        return activations - alpha * direction
    return hook_fn

In [ ]:
def make_neuron_ablation_hook(neuron_idx, alpha):
    def hook_fn(activations, hook):
        activations[..., neuron_idx] *= (1 - alpha)
        return activations
    return hook_fn

In [ ]:
def sae_feature_direction(sae, feature_idx):
    """
    Returns decoder direction for a single SAE feature
    """
    # assumes sae.W_dec: [n_features, d_model]
    return sae.W_dec[feature_idx].detach()

In [ ]:
def run_rewardbench(
    model,
    tokenizer,
    prompts,
    max_new_tokens=256
):
    outputs = []
    for p in prompts:
        tokens = tokenizer(p, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512,do_sample=True,temperature = 0.7,repetition_penalty=1.2, top_p = 0.9)

        text = tokenizer.decode(out[0], skip_special_tokens=True)
        outputs.append(text)
    return outputs

In [ ]:
import pandas as pd

def evaluate_intervention(
    model,
    tokenizer,
    prompts,
    intervention_name,
    eval_fn,
):
    generations = run_rewardbench(model, tokenizer, prompts)

    # eval_fn should return dict per sample
    records = []
    for prompt, gen in zip(prompts, generations):
        metrics = eval_fn(prompt, gen)
        records.append({
            "prompt": prompt,
            "generation": gen,
            **metrics
        })

    df = pd.DataFrame(records)
    df.to_csv(f"{intervention_name}.csv", index=False)
    return df

In [ ]:
from transformer_lens import HookedTransformer

PROMPT_COUNT = 200

def run_full_intervention_sweep(
    model_name,
    tokenizer,
    rewardbench_prompts,
    sae_files,
    top_k_features,
    alpha,
    layer_name,
    eval_fn,
):
    prompts = rewardbench_prompts[:PROMPT_COUNT]

    # ---- Baseline ----
    model = HookedTransformer.from_pretrained(model_name)
    baseline_df = evaluate_intervention(
        model,
        tokenizer,
        prompts,
        "baseline",
        eval_fn
    )

    # ---- SAE interventions ----
    for level, sae_path in enumerate(sae_files):
        sae = torch.load(sae_path, map_location="cuda").eval()

        for k in top_k_features:
            direction = sae_feature_direction(sae, k).to("cuda")

            hook = make_intervention_hook(direction, alpha)

            model.reset_hooks()
            model.add_hook(layer_name, hook)

            name = f"sae_{level}_neuron_{k}"
            evaluate_intervention(
                model,
                tokenizer,
                prompts,
                name,
                eval_fn
            )

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch

def make_sae_clamp_hook(sae, feature_indices, alpha=0.01):
    """
    Returns a hook function that clamps specified SAE features in the residual stream.

    sae: your trained SAE module
    feature_indices: list[int], which features to clamp
    alpha: fraction to remove (0.01 ~ very gentle, 1.0 ~ full zeroing)
    """

    def hook_fn(resid, hook):
        """
        resid: [batch, seq, d_model]
        """
        # Encode to SAE latent space
        z = sae.encoder(resid)      # [batch, seq, d_hidden]
        z_relu = torch.relu(z)       # if you trained with ReLU

        # Prepare delta to remove
        delta = torch.zeros_like(resid)

        # For each feature to clamp
        for i in feature_indices:
            # Extract feature activation
            zi = z_relu[..., i:i+1]             # [batch, seq, 1]
            # Get decoder direction
            direction = sae.decoder.weight[:, i:i+1]  # [d_model, 1]
            # Scale delta
            delta += zi * direction.T           # broadcasting [batch, seq, d_model]

        # Subtract scaled component
        resid_new = resid - alpha * delta

        return resid_new

    return hook_fn


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SAE(nn.Module):
    def __init__(self, d_in, d_hidden, l1_coeff=3e-4):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_hidden, bias=False)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        self.l1_coeff = l1_coeff

        # Tied weights option (recommended for interpretability)
        self.decoder.weight = nn.Parameter(self.encoder.weight.T)

    def forward(self, x):
        # Sparse latent features
        z = F.relu(self.encoder(x))
        # Reconstruction
        x_hat = self.decoder(z)
        return x_hat, z

    def loss(self, x, x_hat, z):
        mse = ((x - x_hat)**2).mean()
        l1 = self.l1_coeff * z.abs().sum(dim=-1).mean()
        return mse + l1


In [ ]:
def append_row(row, path):
    write_header = not os.path.exists(path)
    pd.DataFrame([row]).to_csv(
        path,
        mode="a",
        header=write_header,
        index=False,
    )

In [ ]:
import pandas as pd
import os
import torch # Make sure torch is imported for torch.cuda.empty_cache()
Overall_path = "/content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/"
def clamp_top_neurons_from_csv_and_generate(
    model,
    tokenizer,
    saes,       # list[SAE]
    neurons,  # dict[int -> list of top neurons]
    prompts,
    alpha=1.0,
    max_new_tokens=200,
):

    for sae_idx, sae in enumerate(saes):
      output_path = os.path.join(
          Overall_path,
          f"Modified_outputs_csv{sae_idx+1}_stupid.csv",
      )

      hook_fn = make_sae_clamp_hook(
      sae,                     # ✅ pass SAE module
      feature_indices=neurons[sae_idx],
      alpha=alpha,
)

      hook_name = "blocks.24.hook_resid_post"

      with model.hooks(fwd_hooks=[(hook_name, hook_fn)]):
          for i, prompt in enumerate(prompts):
              tokens = model.to_tokens(prompt, prepend_bos=False)

              with torch.no_grad():
                  out = model.generate(
                      tokens,
                      max_new_tokens=max_new_tokens,
                      do_sample=True,
                      temperature=0.7,
                      top_p=0.9,
                      freq_penalty=1.1,
                  )

              text = model.to_string(out)

              append_row({
                  "alpha": alpha,
                  "prompt": prompt,
                  "response": text,
              }, output_path)

              if i % 10 == 0:
                  torch.cuda.empty_cache()

      print("saved to", output_path)  # reset GPU memory

    # ---- 4. Save CSV ----

In [ ]:
saes = []
for SAE_LEVEL in range(4):
  sae = torch.load(
      f"/content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/sae_mixed{SAE_LEVEL+1}.pt",
      map_location="cuda",
      weights_only=False
  ).eval().to(torch.bfloat16)
  saes.append(sae)

In [ ]:
print(saes[0].state_dict().keys())
print(len(saes))
print(saes[0].type)

odict_keys(['encoder.weight', 'decoder.weight'])
4
<bound method Module.type of SAE(
  (encoder): Linear(in_features=5120, out_features=81920, bias=False)
  (decoder): Linear(in_features=81920, out_features=5120, bias=False)
)>


In [ ]:
Top_k_neurons = []
import pandas as pd

for i in range(4):
  path = f"/content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/rankings/sae_{i}_neuron_ranking.csv"
  df = pd.read_csv(path)
  top_neurons = df['neuron_id'].tolist()[:1]
  Top_k_neurons.append(top_neurons)


[[34083, 21798, 63626], [67829, 10415, 71434], [23765, 4746, 37517], [81629, 47033, 29520]]


In [ ]:
import datasets
from datasets import load_dataset
ds = load_dataset("allenai/reward-bench",split="filtered")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
prompts = []
for i in range(50):
  r = ds[i]
  prompt = r["prompt"]
  prompts.append(prompt)


In [ ]:
for i in range(5):
  print(prompts[i])

How do I detail a car?
Who created the Superman cartoon character?
What is Atlantis?
Hi, I'm in the mood for a Bloody Mary. Can you give me a recipe for making one?
what are some good ways to spread ashes?


In [ ]:
!pip uninstall -y transformer_lens transformers
!pip install transformer_lens transformers # Install latest compatible versions

import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformer_lens
from transformer_lens import HookedTransformer
import torch # Added torch import for torch_dtype

model_name = "EleutherAI/pythia-12b-deduped"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the HuggingFace model
hf_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

# Force-add rotary_pct to the config object if it's missing (should always be 0.5 for Pythia)
hf_model.config.rotary_pct = 0.5 # Pythia models typically have rotary_dim = d_head / 2

# Then pass the pre-loaded and patched HF model to HookedTransformer
model = HookedTransformer.from_pretrained(
    model_name,
    hf_model=hf_model, # Use the pre-loaded and patched HF model
    device="cuda",
    torch_dtype=torch.bfloat16
)

Found existing installation: transformer-lens 2.17.0
Uninstalling transformer-lens-2.17.0:
  Successfully uninstalled transformer-lens-2.17.0
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
  Using cached transformer_lens-2.17.0-py3-none-any.whl.metadata (12 kB)
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
  Using cached transformers-5.1.0-py3-none-any.whl.metadata (31 kB)
  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
Using cached transformer_lens-2.17.0-py3-none-any.whl (195 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

pytorch_model-00003-of-00003.bin:   0%|          | 0.00/4.11G [00:00<?, ?B/s]

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/9.93G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-12b-deduped into HookedTransformer


In [ ]:
import os
for i in range(4):
  file_to_clear = os.path.join(Overall_path, f"Modified_outputs_csv{i+1}_stupid.csv")

  # Open the file in write mode ('w') to clear its content
  with open(file_to_clear, 'w') as f:
      pass

  print(f"File '{file_to_clear}' has been cleared.")

File '/content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/Modified_outputs_csv1_stupid.csv' has been cleared.
File '/content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/Modified_outputs_csv2_stupid.csv' has been cleared.
File '/content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/Modified_outputs_csv3_stupid.csv' has been cleared.
File '/content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/Modified_outputs_csv4_stupid.csv' has been cleared.


In [ ]:
clamp_top_neurons_from_csv_and_generate(
    model,
    tokenizer,
    saes,       # dict[int -> SAE]
    Top_k_neurons,  # dict[int -> path to CSV listing top neurons]
    prompts,
    alpha=0.01,
    max_new_tokens=200,
)


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

...

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/SAE_Works_Pythia-12B/layer 24/Modified_outputs_csv4_stupid.csv


In [ ]:
exit()